<a href="https://colab.research.google.com/github/guddy2005/fake_new_detection_model/blob/main/newmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# Install libraries
!pip install transformers datasets -q

In [22]:
import pandas as pd
import torch
import numpy as np

from transformers import AutoTokenizer, RobertaForSequenceClassification
from transformers import AdamW, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

In [23]:
import pandas as pd

df = pd.read_csv("final_dataset.csv")

# Separate classes into fake (label 0) and real (label 1)
df_fake = df[df["label"] == 0]
df_real = df[df["label"] == 1]

# Find the minimum count between the two classes to balance the dataset
min_count = min(len(df_fake), len(df_real))

# Downsample both classes to the minimum count for a balanced dataset
df_fake = df_fake.sample(min_count, random_state=42)
df_real = df_real.sample(min_count, random_state=42)

# Concatenate the balanced dataframes and shuffle the rows
df = pd.concat([df_fake, df_real]).sample(frac=1, random_state=42)

# Display the value counts for the 'label' column to confirm balancing
print(df["label"].value_counts())

label
1    8460
0    8460
Name: count, dtype: int64


In [24]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [25]:
train_encodings = tokenizer(
    train_texts,
    truncation=True, # Truncate sequences to the maximum length allowed by the model
    padding=True,    # Pad sequences to the maximum length
    max_length=384   # Set the maximum sequence length
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=384
)

In [26]:
# Define a custom PyTorch Dataset for our news data
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # Retrieve an item (tokenized input and label) by index
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        # Return the total number of items in the dataset
        return len(self.labels)

# Create instances of our custom dataset for training and validation
train_dataset = NewsDataset(train_encodings, train_labels)
val_dataset = NewsDataset(val_encodings, val_labels)

In [27]:
from transformers import RobertaForSequenceClassification

# Load the pre-trained RoBERTa model for sequence classification
# We specify num_labels=2 because we have two classes (fake and real news)
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# DataLoaders for training and validation
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

# Optimizer for model parameters
optimizer = AdamW(model.parameters(), lr=1e-5)

epochs = 6

# -------- CLASS WEIGHTS --------
# Compute class weights for unbalanced datasets
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# Loss function with class weights
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

# -------- LR SCHEDULER --------
# Learning rate scheduler with warm-up
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# -------- TRAINING LOOP --------
for epoch in range(epochs):

    print(f"\nEpoch {epoch+1}/{epochs}")

    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):

        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)

        loss = loss_fn(outputs.logits, batch["labels"])

        optimizer.zero_grad()
        loss.backward()

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    print("Training Loss:", total_loss / len(train_loader))

    # -------- VALIDATION --------
    model.eval()

    preds = []
    labels = []

    with torch.no_grad():

        for batch in val_loader:

            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)

            logits = outputs.logits

            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

    acc = accuracy_score(labels, preds)

    print("Validation Accuracy:", acc)

/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(



Epoch 1/6


100%|██████████| 1692/1692 [17:48<00:00,  1.58it/s]


Training Loss: 0.42075203994567556
Validation Accuracy: 0.7420212765957447

Epoch 2/6


100%|██████████| 1692/1692 [17:49<00:00,  1.58it/s]


...

Training Loss: 0.34194055559636805
Validation Accuracy: 0.7579787234042553

Epoch 5/6


100%|██████████| 1692/1692 [17:49<00:00,  1.58it/s]


Training Loss: 0.3369882809061337
Validation Accuracy: 0.7612293144208038

Epoch 6/6


100%|██████████| 1692/1692 [17:49<00:00,  1.58it/s]


Training Loss: 0.32896444403599995
Validation Accuracy: 0.7615248226950354
